# 03.1 — Image and video generation lab

**Prerequisites**

1. You have deployed `gpt-image-1` and `sora` following [README.md](README.md).
2. `.env` contains `MODEL_IMAGE`, `MODEL_VIDEO`, and — if you deployed to a second
   Foundry resource — `AZURE_OPENAI_IMAGE_ENDPOINT` and `AZURE_OPENAI_VIDEO_ENDPOINT`.
3. You hold **Cognitive Services OpenAI User** on every resource involved.

**Cost.** Running this notebook end to end generates roughly six images at
`quality="low"` and one short, small video. Expect $1–3. Every cell that spends
money says so. Do not re-run the video cell out of curiosity.

Everything is written to `lab_output/`, which is git-ignored.

## 1. Setup

The image and video models frequently live on a **different Foundry resource** from
your chat models, because of regional availability. So unlike every previous lab we
cannot simply reuse `chat_client()` — we build clients bound to the right endpoints.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'scripts'))
from ai103 import cfg, credential, chat_client, project_client, token_provider, portal_link

OUT = pathlib.Path.cwd() / 'lab_output'
OUT.mkdir(exist_ok=True)

IMAGE_ENDPOINT = cfg.get('AZURE_OPENAI_IMAGE_ENDPOINT') or cfg.require('AZURE_OPENAI_ENDPOINT')
VIDEO_ENDPOINT = cfg.get('AZURE_OPENAI_VIDEO_ENDPOINT') or cfg.require('AZURE_OPENAI_ENDPOINT')
IMAGE_MODEL = cfg.get('MODEL_IMAGE', 'gpt-image-1')
VIDEO_MODEL = cfg.get('MODEL_VIDEO', 'sora')

print('image endpoint :', IMAGE_ENDPOINT)
print('image model    :', IMAGE_MODEL)
print('video endpoint :', VIDEO_ENDPOINT)
print('video model    :', VIDEO_MODEL)
print('output folder  :', OUT)

The same `token_provider()` helper works here — image generation sits behind the
same `https://cognitiveservices.azure.com/.default` scope as chat. Keyless auth
carries across the whole Foundry surface.

In [ ]:
from openai import AzureOpenAI

images = AzureOpenAI(
    azure_endpoint=IMAGE_ENDPOINT,
    api_version='2025-04-01-preview',   # image edits need a recent preview version
    azure_ad_token_provider=token_provider(),
)
print('image client ready')

A small helper so the rest of the notebook is about the API, not about base64.

Note the single most important line: `item.b64_json`. GPT-image models **always**
return base64. There is no `url` field to fall back on, and `response_format` is
not an accepted parameter.

In [ ]:
import base64, time
from IPython.display import Image as Show, display


def save_result(result, stem, ext='png'):
    """Write every image in an images.generate/edit result to lab_output/."""
    paths = []
    for i, item in enumerate(result.data):
        path = OUT / (f'{stem}.{ext}' if len(result.data) == 1 else f'{stem}_{i}.{ext}')
        path.write_bytes(base64.b64decode(item.b64_json))
        paths.append(path)
        print('wrote', path.name, f'({path.stat().st_size / 1024:.0f} KB)')
    return paths

## 2. Generate an image from a text prompt

**Spends money: one image at `quality="low"`.**

Two things to read carefully in this call:

- `model=` is the **deployment name**, exactly as with chat.
- `quality='low'` is the draft setting. Iterate here. Only when the composition is
  right do you pay for `'high'`.

The prompt describes **content** and **style** separately, because GPT-image does
not have a `style` parameter — DALL·E 3 did, and that is the migration trap.

In [ ]:
PROMPT = (
    'A single ceramic coffee mug on a plain light grey studio backdrop, '
    'seen slightly from above. Soft even product lighting, no props, no text. '
    'Clean commercial product photography.'
)

result = images.generate(
    model=IMAGE_MODEL,
    prompt=PROMPT,
    n=1,
    size='1024x1024',
    quality='low',
    output_format='png',
)

base_path = save_result(result, 'mug_base')[0]
display(Show(filename=str(base_path), width=320))

### The controls, and what each one is for

| Parameter | Values | What it changes | Cost |
|---|---|---|---|
| `size` | `1024x1024`, `1024x1536`, `1536x1024`, `auto` | Resolution **and** aspect ratio | ↑ with pixels |
| `quality` | `low`, `medium`, `high`, `auto` | Rendering effort | ~10× low→high |
| `n` | 1–10 | Variants per call | Linear |
| `output_format` | `png`, `jpeg`, `webp` | Encoding | — |
| `output_compression` | 0–100 | JPEG/WebP quality vs file size | — |
| `background` | `auto`, `transparent`, `opaque` | Alpha channel; `transparent` needs PNG or WebP | — |
| `moderation` | `auto`, `low` | Built-in image moderation strictness | — |

There is no aspect-ratio parameter on `gpt-image-1`: you pick one of three sizes.
Only `gpt-image-2` accepts arbitrary resolutions (edges a multiple of 16 px).

**Spends money: two images.** This cell shows `n`, transparency and JPEG output in
one call — the transparent PNG is the one you want for a product catalogue, because
it composites onto any page background.

In [ ]:
cutout = images.generate(
    model=IMAGE_MODEL,
    prompt='A single ceramic coffee mug, isolated, no background, no shadow, product cutout.',
    n=2,                      # two variants -> twice the cost
    size='1024x1024',
    quality='low',
    background='transparent', # requires png or webp
    output_format='png',
)

cutout_paths = save_result(cutout, 'mug_cutout')

from PIL import Image as PILImage

for p in cutout_paths:
    print(p.name, 'mode =', PILImage.open(p).mode, '(RGBA means a real alpha channel)')

> **Exam note.** `background='transparent'` with `output_format='jpeg'` is a
> contradiction — JPEG has no alpha channel. The service rejects it. Any question
> that pairs transparency with a format is testing this.

## 3. Prompt-driven editing (no mask)

**Spends money: one image.**

`images.edit()` takes the source image and a prompt. Without a mask the model may
change **anything**. That is the right tool for a global restyle and the wrong tool
for a surgical change.

`input_fidelity='high'` tells the model to work harder at preserving the style and
features of the input — particularly faces. It costs latency, not extra images.
It is not supported on `gpt-image-1-mini`.

In [ ]:
with open(base_path, 'rb') as f:
    edited = images.edit(
        model=IMAGE_MODEL,
        image=f,
        prompt='Change the studio backdrop to a warm wooden desk lit by morning window light. '
               'Keep the mug identical in shape and colour.',
        size='1024x1024',
        quality='low',
    )

edited_path = save_result(edited, 'mug_restyled')[0]
display(Show(filename=str(edited_path), width=320))

Compare it to `mug_base.png`. The mug probably drifted a little even though you
asked it not to. That drift is the reason inpainting exists.

## 4. Inpainting with an explicit mask

This is the part of the API people get wrong, so read the rules before the code.

A mask is a **PNG with the same pixel dimensions as the source image**. The model
repaints only the pixels where the mask's **alpha channel is 0** (fully
transparent). Opaque pixels are left alone. The RGB values in the mask are ignored
entirely — a black-and-white bitmap with no alpha channel does nothing useful.

We build the mask programmatically: start fully opaque, then punch a transparent
rectangle into the lower third — the area we want repainted.

In [ ]:
from PIL import Image as PILImage, ImageDraw

source = PILImage.open(base_path).convert('RGBA')
w, h = source.size

# Fully opaque canvas = 'preserve everything'.
mask = PILImage.new('RGBA', (w, h), (0, 0, 0, 255))

# Alpha 0 = 'repaint this'. Lower third of the frame.
ImageDraw.Draw(mask).rectangle([0, int(h * 0.66), w, h], fill=(0, 0, 0, 0))

mask_path = OUT / 'mug_mask.png'
mask.save(mask_path)

print('source :', source.size, source.mode)
print('mask   :', mask.size, mask.mode, '  <- must match exactly, and must be RGBA')
transparent = sum(1 for px in mask.getdata() if px[3] == 0)
print(f'editable region: {transparent / (w * h):.0%} of the image')

**Spends money: one image.**

The prompt for a masked edit should describe **the whole desired image**, not just
the patch. The model uses the unmasked pixels as context and the prompt as the
target description.

In [ ]:
with open(base_path, 'rb') as img_f, open(mask_path, 'rb') as mask_f:
    inpainted = images.edit(
        model=IMAGE_MODEL,
        image=img_f,
        mask=mask_f,
        prompt='The same ceramic mug, now standing on a slab of dark slate with a few '
               'scattered coffee beans in front of it.',
        size='1024x1024',
        quality='low',
    )

inpainted_path = save_result(inpainted, 'mug_inpainted')[0]
display(Show(filename=str(inpainted_path), width=320))

The top two-thirds should be pixel-similar to the original; only the masked band
changed. That is the difference between an edit and an inpaint, and it is the
reason production pipelines mask.

### Choosing between the three edit modes

| You want to | Use | Inputs |
|---|---|---|
| Restyle the whole picture | Prompt-driven edit | `image` + `prompt` |
| Change one region, preserve the rest | Inpainting | `image` + `mask` + `prompt` |
| Get alternatives to a composition | Variation | `image`, weak prompt |
| Combine subjects from several photos | Multi-reference edit | `image=[f1, f2]` + `prompt` |

## 5. Content filtering on the image path

**Free — this call is expected to be rejected before any image is rendered.**

Two distinct failures exist and the error message tells you which:

- *"Your task failed as a result of our safety system"* — the **prompt** was blocked.
- *"Generated image was filtered as a result of our safety system"* — the prompt
  passed and the **rendered output** was blocked. You are billed for that one.

Unit 03.3 configures these filters properly.

In [ ]:
from openai import BadRequestError

try:
    images.generate(
        model=IMAGE_MODEL,
        prompt='A photorealistic image of a named living politician endorsing a product.',
        n=1,
        size='1024x1024',
        quality='low',
    )
    print('not blocked — filters are probabilistic, try a different prompt')
except BadRequestError as e:
    print('blocked, as designed')
    print('code   :', getattr(e, 'code', None))
    print('detail :', str(e)[:400])

## 6. Video generation — the async job pattern

**Spends money: one short video. Roughly $0.50–1.00. Run this cell once.**

There is no synchronous video API. Every Sora integration has the same three steps,
and the shape of your application follows from that:

```
POST  /openai/v1/video/generations/jobs           -> { id, status: 'queued' }
GET   /openai/v1/video/generations/jobs/{id}      -> poll until terminal
GET   /openai/v1/video/generations/{gen}/content/video -> MP4 bytes
```

We call it with `requests` and an Entra ID bearer token rather than an SDK, because
the job API is the thing the study guide describes and it is stable across client
versions. `api-version=preview` is correct for the preview period.

Terminal states are `succeeded`, `failed`, `cancelled`. `failed` is **normal** —
content filtering rejects prompts and rendered frames — so read `failure_reason`
rather than raising.

In [ ]:
import requests

API_VERSION = 'preview'
token = credential().get_token('https://cognitiveservices.azure.com/.default').token
headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

# 1. Submit
create_url = f'{VIDEO_ENDPOINT}/openai/v1/video/generations/jobs?api-version={API_VERSION}'
body = {
    'model': VIDEO_MODEL,          # deployment name
    'prompt': 'A slow dolly shot across a wooden workbench covered in hand tools, '
              'warm lamplight, shallow depth of field.',
    'width': 480,                  # smallest sensible size -> cheapest
    'height': 480,
    'n_seconds': 5,                # billed per second of output
    'n_variants': 1,               # 1-4; each variant is billed
}

resp = requests.post(create_url, headers=headers, json=body, timeout=60)
resp.raise_for_status()
job = resp.json()
job_id = job['id']
print('job', job_id, '->', job.get('status'))

In [ ]:
# 2. Poll. 5-10 seconds is a reasonable interval; sub-second polling just earns 429s.
status_url = f'{VIDEO_ENDPOINT}/openai/v1/video/generations/jobs/{job_id}?api-version={API_VERSION}'

status, state = None, {}
started = time.time()
while status not in ('succeeded', 'failed', 'cancelled'):
    if time.time() - started > 600:
        raise TimeoutError('gave up after 10 minutes')
    time.sleep(5)
    state = requests.get(status_url, headers=headers, timeout=60).json()
    status = state.get('status')
    print(f'{time.time() - started:6.0f}s  {status}')

if status != 'succeeded':
    print('failure_reason:', state.get('failure_reason'))

In [ ]:
# 3. Retrieve. Generated videos expire in Azure OpenAI storage - download to keep them.
video_path = None
if status == 'succeeded':
    generations = state.get('generations', [])
    generation_id = generations[0]['id']
    content_url = (
        f'{VIDEO_ENDPOINT}/openai/v1/video/generations/{generation_id}'
        f'/content/video?api-version={API_VERSION}'
    )
    mp4 = requests.get(content_url, headers=headers, timeout=300)
    mp4.raise_for_status()
    video_path = OUT / 'workbench.mp4'
    video_path.write_bytes(mp4.content)
    print('wrote', video_path, f'({video_path.stat().st_size / 1e6:.1f} MB)')
    print('generation id:', generation_id)
else:
    print('nothing to download')

In [ ]:
from IPython.display import Video

if video_path and video_path.exists():
    display(Video(str(video_path), embed=True, width=360))

> **Exam note.** The architectural consequence of the async pattern is that the
> request that *starts* a render must not be the request that *serves* the user.
> Return the job ID and poll from the client, or complete the job on a queue worker
> and notify. A synchronous web handler that blocks on a five-minute render is the
> wrong answer to every version of this question.

## 7. Reference media and video editing

**These cells do not run by default — they would each cost another render.** Read
them, then enable one if you want to see it work.

### Generating from a reference image (Sora job API)

The job API takes reference media as `inpaint_items`, sent as `multipart/form-data`
alongside the JSON body. `crop_bounds` selects the part of the image to use, as a
fraction from each edge, and `frame_index` chooses which frame of the output the
image anchors (0 = the first frame).

In [ ]:
RUN_REFERENCE_VIDEO = False  # flip to True to spend another render

if RUN_REFERENCE_VIDEO:
    import json

    ref = base_path  # the mug image we generated earlier
    form_headers = {'Authorization': f'Bearer {token}'}  # no Content-Type: requests sets the boundary
    payload = {
        'model': VIDEO_MODEL,
        'prompt': 'The camera slowly orbits the mug as steam begins to rise.',
        'height': '480',
        'width': '480',
        'n_seconds': '5',
        'n_variants': '1',
        'inpaint_items': json.dumps([
            {
                'frame_index': 0,
                'type': 'image',
                'file_name': ref.name,
                'crop_bounds': [0.0, 0.0, 0.0, 0.0],  # left, top, right, bottom fractions
            }
        ]),
    }
    files = [('files', (ref.name, ref.open('rb'), 'image/png'))]
    r = requests.post(create_url, headers=form_headers, data=payload, files=files, timeout=120)
    print(r.status_code, r.text[:500])
else:
    print('skipped - set RUN_REFERENCE_VIDEO = True to run it')

### The Sora 2 client surface

If your deployment is `sora-2`, the same three steps exist as SDK methods on the
OpenAI v1 client. Different parameter names — this is the table to memorise:

| Concept | Sora job API | Sora 2 client |
|---|---|---|
| Duration | `n_seconds` (int) | `seconds='4' \| '8' \| '12'` |
| Resolution | `width` + `height` | `size='720x1280' \| '1280x720'` |
| First-frame reference | `inpaint_items` + `crop_bounds` | `input_reference=<file>` (must match `size` exactly) |
| Poll | `GET .../jobs/{id}` | `client.videos.retrieve(id)` |
| Download | `GET .../content/video` | `client.videos.download_content(id, variant='video')` |
| Edit an existing video | resubmit with the clip as a reference | `client.videos.remix(video_id, prompt)` |

Both are **preview**.

In [ ]:
RUN_SORA2 = False  # requires a sora-2 deployment and a recent `openai` package

if RUN_SORA2:
    from openai import OpenAI

    v1 = OpenAI(base_url=f'{VIDEO_ENDPOINT}/openai/v1/', api_key=token_provider())

    video = v1.videos.create(
        model='sora-2',
        prompt='A slow dolly across a workbench covered in hand tools, warm lamplight.',
        size='1280x720',
        seconds=4,
    )
    while video.status not in ('completed', 'failed'):
        time.sleep(5)
        video = v1.videos.retrieve(video.id)
        print(video.status, video.progress)

    if video.status == 'completed':
        v1.videos.download_content(video.id, variant='video').write_to_file(OUT / 'sora2.mp4')

        # Editing = remix. Change ONE thing per call; multiple simultaneous
        # instructions degrade fidelity to the source.
        remix = v1.videos.remix(
            video_id=video.id,
            prompt='Shift the colour palette to cool blue evening light.',
        )
        print('remix job:', remix.id)
else:
    print('skipped - set RUN_SORA2 = True if you deployed sora-2')

### The practical video-editing workflow

Generative video gives you weak control over any individual frame. The reliable
pattern moves the precision to where you have it:

1. **Generate a still** with `gpt-image-1` and iterate at `quality='low'` until the
   composition is exactly right — cheap, fast, fully controllable.
2. **Inpaint** any detail that is still wrong. Now frame 0 is exactly what you want.
3. **Animate** it: pass the still as the reference/first frame to Sora.
4. **Remix** for one narrow change at a time if the motion needs adjusting.
5. **Post-process deterministically** — trim, concatenate, overlay a watermark —
   with ffmpeg. Never ask a generative model to do something a codec can do.

## 8. What you spent

There is no `usage` object on an image or video response — these are billed per
asset, not per token, so you have to count assets yourself. Do this in production
too: a per-request counter and a hard cap are the only things standing between a
bug and a large invoice.

In [ ]:
pngs = sorted(OUT.glob('*.png'))
mp4s = sorted(OUT.glob('*.mp4'))
print(f'{len(pngs)} images, {len(mp4s)} videos in {OUT.name}')
for p in pngs + mp4s:
    print(f'  {p.name:<24} {p.stat().st_size / 1024:8.0f} KB')
print('\nBilling model: images per image (size x quality), video per second x resolution.')
print('Cost check:', portal_link('cost'))

## Exercise

Solutions are in [quiz.md](quiz.md).

1. **Quality is not composition.** Regenerate `PROMPT` at `quality='high'` and at
   `size='1536x1024'`. Save both. Write two sentences on what actually changed
   between low and high, and argue whether the extra cost belongs in an iteration
   loop or only in the final render.

2. **A precise mask.** Build a mask that makes only a **circle in the centre** of
   the image editable, and inpaint a different object into it. Assert in code that
   the mask is the same size as the source and is `RGBA` before you send it.

3. **A reusable job runner.** Write `generate_video(prompt, seconds, width, height,
   poll_every=5, timeout=600) -> pathlib.Path | None` that submits, polls, handles
   all three terminal states, returns `None` with the `failure_reason` printed on
   failure, and never spins faster than `poll_every`.

4. **Cost guard.** Wrap `images.generate` in a function that refuses to run if the
   estimated spend for the session would exceed a budget you pass in. Use a simple
   price table keyed on `(size, quality)` and count `n`.

Use the empty cell below.

## Clean up

The cell below lists what you must delete. Deployment deletion is a control-plane
operation, so it is deliberately left as an explicit portal or CLI step rather than
something a notebook does behind your back.

**Do not skip this.** Image and video deployments hold scarce regional quota.

In [ ]:
rg = cfg.get('AZURE_RESOURCE_GROUP', 'rg-ai103-lab')
print('Delete these before you move on:\n')
for dep, hint in ((IMAGE_MODEL, 'image'), (VIDEO_MODEL, 'video')):
    print(f'  az cognitiveservices account deployment delete -g {rg} '
          f'-n <{hint}-resource> --deployment-name {dep}')
print('\nOr: https://ai.azure.com -> your project -> Deployments -> select -> Delete')
print('If you created extra Foundry resources, delete AND purge them as well.')
print(f'\nGenerated media stays in {OUT} (git-ignored). Delete it yourself if you want the disk back.')